# PaperQA2 baseline benchmark (30 questions)

Second baseline for reviewer 1, major comment 2. PaperQA2
(https://github.com/future-house/paper-qa) is a published RAG system built specifically
for full-text scientific papers, which makes it a closer comparator to the T21 Research
Assistant than MedRAG: MedRAG's native corpora are abstracts and textbooks, whereas
PaperQA2 is designed to read whole papers and produce a cited answer.

Held constant against the T21 arm:

- Same documents: the 1,594 article-only papers in the live Milvus collection
  (DS_INCLUDE_2026_09_02_DEEPINFRA_2048, 65,677 vectors; fresh round 2026-09-06),
  written as one text file per paper by the baseline-corpora notebook.
- Same generator: nemotron-3-super via the Ollama Cloud OpenAI-compatible endpoint.
  PaperQA2 has THREE LLM roles (llm, summary_llm, agent_llm) and ships with GPT-4o as
  the default for all of them; all three are repointed, otherwise part of the answer
  quality would come from a different model.
- Same scoring: the sentence-level adapted BERTScore from notebook 3, computed through
  the production /embed_batch endpoint.

What differs, and is therefore what the comparison measures (all values read from the
live settings of both systems):

- Chunking: T21 cuts at 1200 characters with 200 overlap and labels each chunk with its
  section; PaperQA2 cuts at 5000 characters with 250 overlap and keeps no section label.
- Retrieval: T21 searches Results, Discussion, Methods and Conclusion separately so each
  gets equal candidate representation, and excludes Abstract and Introduction entirely;
  PaperQA2 runs one flat dense search over everything.
- Ranking: T21 reranks with a cross-encoder and multiplies by a section priority
  (RERANK_ALPHA 0.8); PaperQA2 has no reranking stage.
- Evidence handling: PaperQA2 summarises each of its 12 retrieved chunks with the LLM
  into a relevance-scored note, drops the low scorers, then answers from at most 8 of
  them. T21 passes retrieved text into a single synthesis call. This is why PaperQA2
  issues about 16 LLM calls per question where MedRAG issues one, and it is a genuine
  advantage PaperQA2 has, not merely a difference.
- Output: T21 caps the summary at 150 words and cites paper plus section on every
  sentence; PaperQA2 targets about 200 words with no hard cap and cites by paper only.

Note that PaperQA2 has no reranking stage, so T21's third model (the
llama-nemotron-rerank cross-encoder) has no counterpart here. "Same models" holds for
the generator and, with EMBED_PROVIDER="t21", the embedder, but not for the full stack.

Environment: run with the `paperqa` kernel (Python 3.11), not `dschat` or `medrag`.

    conda run -n paperqa python -m ipykernel install --user --name paperqa

Cost and runtime, measured on this setup: indexing embeds about 13.6M tokens through
text-embedding-3-small (well under a dollar) and takes tens of minutes; each question
then costs roughly 12 to 16 LLM calls, because PaperQA2 summarises each retrieved chunk
separately before writing the answer. Budget noticeably more time than MedRAG, which
used a single LLM call per question.


## T21 Research Assistant vs PaperQA2: what is compared

Held constant in the benchmark (with EMBED_PROVIDER="t21" this arm is a tight ablation:
same corpus, same embedder, same generator, so only the architecture differs):

| Aspect | Both arms |
|---|---|
| Corpus | Same 1,594 article-only papers (paperqa_corpus_2026-09-02_articles, matching the live collection exactly), one plain-text file per paper; earlier runs used the superseded 1,602- and 1,765-paper sets |
| Generator | nemotron-3-super via the Ollama Cloud OpenAI-compatible endpoint; all three PaperQA2 LLM roles (llm, summary_llm, agent_llm) repointed from the GPT-4o default |
| Embedder | llama-nemotron-embed-vl-1b-v2, 2048-dim, via DeepInfra (EMBED_PROVIDER="t21"; verified by HTTP capture, no OpenAI traffic) |
| Questions | Same 30 positive-control questions with expert reference answers |
| Scoring | Same sentence-level adapted BERTScore + semantic similarity via the production /embed_batch endpoint; citation tags stripped from both arms before scoring |

What differs (the thing under test):

| Dimension | T21 Research Assistant | PaperQA2 baseline |
|---|---|---|
| Chunking | 1,200 chars, 200 overlap, each chunk labeled with its section | 5,000 chars, 250 overlap, no section labels |
| Section awareness | Per-section retrieval over Results, Discussion, Methods, Conclusion (12 candidates each); Abstract and Introduction excluded entirely | One flat dense search over everything |
| Reranking | Cross-encoder (llama-nemotron-rerank-vl-1b-v2) blended with vector score at alpha 0.8, then section priority multiplier | None (T21's third model has no counterpart here) |
| Evidence handling | Retrieved text passed into a single synthesis call | Each of 12 retrieved chunks summarised by the LLM into a relevance-scored note; low scorers dropped; answers from at most 8 |
| LLM calls per question | 2 to 4 (synthesis plus judge/improve) | ~16 (per-chunk evidence summarisation plus synthesis) |
| Paper diversity | Max 2 chunks per PMCID | None enforced |
| Vector store | Milvus IVF_FLAT, inner product, nprobe 64 | In-memory Docs index (pickled) |
| Pipeline | 8-node LangGraph: question judge, retrieve, rerank, synthesize, raw evidence, safety guard, finalize, judge-improve | Non-agentic Docs pipeline: chunk, embed, retrieve, summarise evidence, answer (agentic loop unusable: paper-qa 2026.8.12 vs fhlmi 1.0.7) |
| Question validation | Pre-flight VALID/INVALID classifier; evidence-gap refusal path | None; always answers |
| Output format | Structured Summary (150-word cap) + Key Evidence + Reasoning; every sentence cited as paper plus section, INCLUDE badges | Free text, ~200-word target with no hard cap, cites by paper only ("(Smith2020 chunk 3)") |
| Safety layer | Prescription/directive language filter | None |

Measured scores (SUPERSEDED: provisional run on mismatched corpora; the fresh 2026-09-06 same-corpus run on the 1,594-article set replaces these numbers). Corpus
asymmetry caveat: in this run T21 answered on the v6 corpus and PaperQA2/MedRAG on the
v7 1,602-paper build; treat as indicative until the clean same-corpus rerun replaces
them.

| Metric | T21 | PaperQA2 | MedRAG |
|---|---|---|---|
| BERTScore P | 0.556 | 0.529 | 0.540 |
| BERTScore R | 0.565 | 0.610 | 0.614 |
| BERTScore F | 0.556 | 0.561 | 0.569 |
| BERTScore F (size-matched) | - | 0.602 | - |
| Semantic similarity | 0.636 | 0.683 | 0.685 |
| Mean answer length | 101 words | 163 words | 252 words |

Reading note: like MedRAG, PaperQA2's higher recall and semantic similarity track its
longer answers; the size-matched scoring in cell 5b controls for that. Its size-matched
F rising above the full-length F reflects trimming to the most reference-relevant
sentences, which mostly lifts precision.

In [1]:
# ---- Cell 1: configuration ----
import json, os, pickle, re, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

NB_DIR   = Path.cwd()
ANALYSIS = NB_DIR.parent if NB_DIR.name == "nbs" else NB_DIR
BENCH    = ANALYSIS / "benchmarks"
# Which corpus build to run against: "2026-09-02_articles" is the 1,594-paper
# article-only corpus from the baseline-corpora notebook (fresh round 2026-09-06),
# matching the live collection exactly.
CORPUS_DATE = "2026-09-02_articles"
CORPUS   = BENCH / (f"paperqa_corpus_{CORPUS_DATE}" if CORPUS_DATE else "paperqa_corpus")
OUT_DIR  = ANALYSIS / "output" / "paperqa_benchmark"; OUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS_XLSX = ANALYSIS / "input" / "Revised_DownSyndrome_60_Questions_Final_linked.xlsx"
T21_EMBED_BATCH_URL = "http://localhost:8698/embed_batch"

# Which embedder PaperQA2 retrieves with:
#   "openai" keeps its shipped text-embedding-3-small, so the arm is PaperQA2 as
#     published, answering how an off-the-shelf scientific RAG system does on this corpus.
#   "t21" substitutes the exact embedder the T21 pipeline uses
#     (nvidia/llama-nemotron-embed-vl-1b-v2, 2048-dim, via DeepInfra). Combined with the
#     shared generator the arms then share corpus, embedder and LLM, so the comparison
#     isolates architecture. Verified by capturing the HTTP calls: with "t21" the only
#     traffic is ollama.com (model=nemotron-3-super) and DeepInfra embeddings, no OpenAI.
# The two indexes are not interchangeable: switching requires deleting paperqa_docs.pkl.
EMBED_PROVIDER    = "t21"

EVIDENCE_K        = 12     # chunks retrieved per question
ANSWER_MAX_SOURCE = 8      # sources allowed in the final answer
INDEX_LIMIT       = None   # set to e.g. 50 for a fast smoke test, None for all 1,594

sys.path.insert(0, str(BENCH))
print("corpus dir  :", CORPUS, "|", len(list(CORPUS.glob("*.txt"))), "papers")
print("questions   :", QUESTIONS_XLSX.name, "| exists:", QUESTIONS_XLSX.exists())
print("output dir  :", OUT_DIR)

# ── Live collection gate (fresh-results round 2026-09-06) ─────────────────────
EXPECTED_COLLECTION = "DS_INCLUDE_2026_09_02_DEEPINFRA_2048"
EXPECTED_VECTORS, EXPECTED_DOCS = 65_677, 1_594
_hj = requests.get("http://localhost:8698/health", timeout=30).json()
_hc = _hj["config"]["milvus_collection"]
_hv = int(str(_hj["database"]["total_vectors"]).replace(",", ""))
_hd = int(str(_hj["database"]["estimated_documents"]).replace(",", ""))
assert (_hc, _hv, _hd) == (EXPECTED_COLLECTION, EXPECTED_VECTORS, EXPECTED_DOCS), (
    f"API serves {_hc} ({_hv:,} vectors / {_hd:,} docs); expected "
    f"{EXPECTED_COLLECTION} ({EXPECTED_VECTORS:,} / {EXPECTED_DOCS:,})")
print(f"Collection gate OK: {_hc} ({_hv:,} vectors, {_hd:,} documents)")



corpus dir  : /path/to/project/T21_research_assistant_v7/analysis/benchmarks/paperqa_corpus_2026-09-02_articles | 1594 papers
questions   : Revised_DownSyndrome_60_Questions_Final_linked.xlsx | exists: True
output dir  : /path/to/project/T21_research_assistant_v7/analysis/output/paperqa_benchmark
Collection gate OK: DS_INCLUDE_2026_09_02_DEEPINFRA_2048 (65,677 vectors, 1,594 documents)


## Cell 2: the 30 positive-control questions

Same set as notebook 17, so the two baselines are scored on identical questions against
identical reference answers. The 30 negative controls are excluded here because they
have no reference answer to score against; refusal behaviour is evaluated separately.


In [2]:
# ---- Cell 2: questions ----
df_q = pd.read_excel(QUESTIONS_XLSX, sheet_name="Questions_for_testing")
df_q.columns = [c.strip() for c in df_q.columns]
pos = df_q[df_q["CTRL_Type"].str.strip().str.lower() == "positive"].copy().reset_index(drop=True)
pos = pos.rename(columns={"Original question": "question", "Answer": "ground_truth"})
pos["question"] = pos["question"].astype(str).str.strip()
pos["ground_truth"] = pos["ground_truth"].astype(str).str.strip()
assert len(pos) == 30, f"expected 30 positive controls, found {len(pos)}"
print(f"positive controls: {len(pos)}")
print(f"reference answers average {pos['ground_truth'].str.split().str.len().mean():.0f} words")
pos[["SLNo", "question"]].head(5)


positive controls: 30
reference answers average 76 words


,SLNo,question
0,Q1,Which clinical conditions are people with Down...
1,Q2,Which clinical conditions are people with Down...
2,Q3,Why do people with Down syndrome tend to have ...
3,Q4,Why are people with Down syndrome at high risk...
4,Q5,Do individuals with Down syndrome have overact...


## Cell 3: index the corpus

PaperQA2's agentic entry point (`ask`) is unusable in this environment: paper-qa
2026.8.12 calls `LiteLLMModel.get_router()`, which fhlmi 1.0.7 removed, so the
ToolSelector agent raises AttributeError. This notebook therefore uses PaperQA2's core
Docs API, which chunks, embeds, retrieves, summarises evidence per chunk and synthesises
a cited answer.

That is also the better choice for a benchmark. The agent issues a variable number of
searches per question, which would add run-to-run variance to an arm that is supposed to
be a fixed comparator. State this deviation in the manuscript: it is PaperQA2's
retrieval and answering pipeline run deterministically, not its agentic search loop.

The built index is pickled, so re-running the notebook does not repeat the embedding
cost. Delete `paperqa_docs.pkl` to force a rebuild.


In [3]:
# ---- Cell 3: build or load the PaperQA2 index ----
from paperqa_adapter import make_settings, build_docs, query_docs, answer_text

settings = make_settings(paper_directory=CORPUS,
                         index_directory=OUT_DIR / "pqa_index",
                         evidence_k=EVIDENCE_K,
                         answer_max_sources=ANSWER_MAX_SOURCE,
                         embed_provider=EMBED_PROVIDER)
print("llm       :", settings.llm)
print("summary   :", settings.summary_llm)
print("agent_llm :", settings.agent.agent_llm)
print("embedding :", settings.embedding)
print("corpus    :", CORPUS, "|", len(list(CORPUS.glob("*.txt"))), "papers")

DOCS_PKL = OUT_DIR / (f"paperqa_docs_{CORPUS_DATE}.pkl" if CORPUS_DATE else "paperqa_docs.pkl")
if DOCS_PKL.exists():
    t0 = time.time()
    docs = pickle.loads(DOCS_PKL.read_bytes())
    print(f"loaded cached index in {time.time()-t0:.0f}s")
else:
    t0 = time.time()
    docs = build_docs(settings, paper_directory=CORPUS, limit=INDEX_LIMIT, progress_every=100)
    DOCS_PKL.write_bytes(pickle.dumps(docs))
    print(f"indexed and cached in {time.time()-t0:.0f}s")
print("documents in index:", len(getattr(docs, "docs", {})))



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

llm       : openai/nemotron-3-super
summary   : openai/nemotron-3-super
agent_llm : openai/nemotron-3-super
embedding : openai/nvidia/llama-nemotron-embed-vl-1b-v2
corpus    : /path/to/project/T21_research_assistant_v7/analysis/benchmarks/paperqa_corpus_2026-09-02_articles | 1594 papers
  indexed 100/1594 (ok 100, failed 0)
  indexed 200/1594 (ok 200, failed 0)
  indexed 300/1594 (ok 300, failed 0)
  indexed 400/1594 (ok 400, failed 0)
  indexed 500/1594 (ok 500, failed 0)
  indexed 600/1594 (ok 600, failed 0)
  indexed 700/1594 (ok 700, failed 0)
  indexed 800/1594 (ok 800, failed 0)
  indexed 900/1594 (ok 900, failed 0)
  indexed 1000/1594 (ok 1000, failed 0)
  indexed 1100/1594 (ok 1100, failed 0)


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x74853ff820d0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x74851b072dd0>, 13501791.35439892), (<aiohttp.client_proto.ResponseHandler object at 0x74851613bc50>, 13501792.399452504), (<aiohttp.client_proto.ResponseHandler object at 0x748521f546d0>, 13501810.966678314), (<aiohttp.client_proto.ResponseHandler object at 0x74851d75f3d0>, 13501811.741708769), (<aiohttp.client_proto.ResponseHandler object at 0x74851c375f50>, 13501817.815661712)])']
connector: <aiohttp.connector.TCPConnector object at 0x74853ff81e50>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x74853e62aa10>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x74853da71450>, 13501780.60279602), (<aiohttp.client_proto.ResponseHandler object at 0x74853e6170d0>, 13501783.02404516), (<aiohttp.client_proto.ResponseHandler object at

  indexed 1200/1594 (ok 1200, failed 0)
  indexed 1300/1594 (ok 1300, failed 0)
  indexed 1400/1594 (ok 1400, failed 0)
  indexed 1500/1594 (ok 1500, failed 0)
PaperQA2 corpus indexed: 1594 papers (0 failed)
indexed and cached in 5020s
documents in index: 1594


## Cell 4: run the 30 questions

Checkpointed to `paperqa_answers.json`, so an interrupted run resumes rather than
repeating LLM calls. The contexts PaperQA2 used are stored alongside each answer so any
answer can be traced back to the papers that supported it.


In [4]:
# ---- Cell 4: PaperQA2 answers ----
# Corpus-specific checkpoint, so answers from one corpus are never reused for another.
ANS_PATH = OUT_DIR / (f"paperqa_answers_{CORPUS_DATE}.json" if CORPUS_DATE else "paperqa_answers.json")
answers = json.loads(ANS_PATH.read_text()) if ANS_PATH.exists() else {}

for i, row in pos.iterrows():
    key = str(row["SLNo"])
    if key in answers and answers[key].get("answer"):
        continue
    t0 = time.time()
    try:
        resp = query_docs(docs, row["question"], settings)
        sess = getattr(resp, "session", resp)
        ctxs = getattr(sess, "contexts", []) or []
        answers[key] = {
            "question": row["question"],
            "answer": answer_text(resp),
            "secs": round(time.time() - t0, 1),
            "papers": sorted({getattr(getattr(c, "text", None), "doc", None) and
                              str(getattr(getattr(c, "text").doc, "docname", "")) for c in ctxs} - {""}),
            "n_contexts": len(ctxs),
        }
    except Exception as exc:
        answers[key] = {"question": row["question"], "answer": "", "error": str(exc)[:200]}
        print(f"  [{key}] ERROR {str(exc)[:110]}")
    ANS_PATH.write_text(json.dumps(answers, indent=1))
    print(f"  [{i+1}/{len(pos)}] {answers[key].get('secs','-')}s  {row['question'][:60]}")

pos["paperqa_answer"] = [answers.get(str(s), {}).get("answer", "") for s in pos["SLNo"]]
print(f"\nanswered: {int((pos['paperqa_answer'].str.len() > 0).sum())}/{len(pos)}")
print(f"mean answer length: {pos['paperqa_answer'].str.split().str.len().mean():.0f} words")


  [1/30] 61.7s  Which clinical conditions are people with Down syndrome at h
  [2/30] 79.7s  Which clinical conditions are people with Down syndrome at l
  [3/30] 57.5s  Why do people with Down syndrome tend to have hyperactive im
  [4/30] 44.3s  Why are people with Down syndrome at high risk of developing
  [5/30] 70.6s  Do individuals with Down syndrome have overactive thyroid gl
  [6/30] 60.7s  Which interferon receptors are triplicated in Down syndrome?
  [7/30] 22.4s  How is Stat1 signaling affected in Down syndrome?
  [8/30] 53.0s  Which circulating immune cell is consistently depleted in T2


acompletion attempt failed on openai/nemotron-3-super.
Traceback (most recent call last):
  File "/home/user/anaconda3/envs/paperqa/lib/python3.11/site-packages/litellm/llms/custom_httpx/aiohttp_transport.py", line 61, in map_aiohttp_exceptions
    yield
  File "/home/user/anaconda3/envs/paperqa/lib/python3.11/site-packages/litellm/llms/custom_httpx/aiohttp_transport.py", line 307, in handle_async_request
    response = await self._make_aiohttp_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/user/anaconda3/envs/paperqa/lib/python3.11/site-packages/litellm/llms/custom_httpx/aiohttp_transport.py", line 285, in _make_aiohttp_request
    response = await client_session.request(**request_kwargs).__aenter__()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/user/anaconda3/envs/paperqa/lib/python3.11/site-packages/aiohttp/client.py", line 1693, in __aenter__
    self._resp: _RetType_co = await self._coro
                         

  [9/30] 246.1s  What are some advantages and caveats of the Ts65Dn and Dp16 
  [10/30] 43.9s  Are individuals with Down syndrome more prone to high blood 
  [11/30] 35.8s  What neurodegeneration is associated with Down syndrome?
  [12/30] 71.3s  What type of cancer are individuals with Down Syndrome less 
  [13/30] 46.1s  Are vaccines safe and efficacious in people with Down syndro
  [14/30] 35.4s  Do individuals with Down syndrome have altered liver functio
  [15/30] 34.0s  What is the cause of Down syndrome regression disorder?
  [16/30] 28.2s  How common is Down syndrome at birth in the United States, a
  [17/30] 34.8s  Which congenital heart defects are most common in Down syndr
  [18/30] 52.7s  Why are children with Down syndrome at high risk of acute me
  [19/30] 61.0s  What is transient myeloproliferative disorder in Down syndro
  [20/30] 115.3s  Which cancers are more common and which are less common in i
  [21/30] 59.7s  How common is obstructive sleep apnea in children with 

## Cell 5: sentence-level adapted BERTScore

Identical implementation to notebooks 3 and 17: same sentence splitter including the
semicolon rule for the key-point style of the expert answers, same embedder, same
precision, recall and F definitions. Reusing it verbatim is what makes the PaperQA2
numbers comparable to the MedRAG, GPT, Claude and Nemotron numbers.

PaperQA2 writes inline citation keys such as "(Smith2020 chunk 3)". These are stripped
before scoring, exactly as the T21 arm strips its `[PMCxxxxxxx:Section]` tags, so no arm
is penalised for the surface form of its citations.


In [5]:
# ---- Cell 5: scoring ----
_SENT_RE_SC = re.compile(r"(?<=[.!?])\s+")
_SEMI_RE_SC = re.compile(r"\s*;\s*")
_PQA_CITE_RE = re.compile(r"\(([^()]*?(?:chunk|pages?)\s*\d+[^()]*?)\)", re.I)

_ERR_MARKER_RE_SC = re.compile(r"\[[A-Z][A-Z0-9_]*_ERROR\]|too many concurrent", re.I)

def _is_invalid_answer_sc(text) -> bool:
    """Empty or API-error answers are EXCLUDED from scoring (NaN), never scored as 0.
    Genuine short model outputs ("I cannot answer.") are NOT invalid and stay in."""
    t = str(text or "").strip()
    return (not t) or t.lower() == "nan" or bool(_ERR_MARKER_RE_SC.search(t))


def _strip_citations(text):
    """Remove inline citation keys so scoring compares prose, not citation style."""
    t = _PQA_CITE_RE.sub("", str(text or ""))
    t = re.sub(r"\[PMC[^\]]+\](?:\([^)]*\))?", "", t)
    return re.sub(r"\s+", " ", t).strip()

def _split_sents_sc(text):
    raw = _SENT_RE_SC.split(str(text or "").strip())
    result = []
    for chunk in raw:
        chunk = chunk.strip()
        if not chunk:
            continue
        if ";" in chunk and not re.search(r"[.!?]\s*$", chunk):
            result.extend([p.strip() for p in _SEMI_RE_SC.split(chunk) if p.strip()])
        else:
            result.append(chunk)
    return result

def _embed_batch_sc(texts):
    r = requests.post(T21_EMBED_BATCH_URL, json={"texts": texts, "mode": "document"}, timeout=180)
    r.raise_for_status()
    return np.array(r.json()["embeddings"], dtype=np.float32)

def _bertscore_sc(reference, candidate):
    # NaN (excluded from means), not 0: empty/unsplittable input means "cannot score".
    _default = {"bertscore_p": float("nan"), "bertscore_r": float("nan"),
                "bertscore_f": float("nan")}
    ref_sents, cand_sents = _split_sents_sc(reference), _split_sents_sc(candidate)
    if not ref_sents or not cand_sents:
        return _default
    try:
        vecs = _embed_batch_sc(ref_sents + cand_sents)
        vecs = vecs / np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-9)
        sim = vecs[:len(ref_sents)] @ vecs[len(ref_sents):].T
        recall = float(sim.max(axis=1).mean())
        precision = float(sim.max(axis=0).mean())
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall > 0 else 0.0
        return {"bertscore_p": round(precision, 4), "bertscore_r": round(recall, 4),
                "bertscore_f": round(f1, 4)}
    except Exception as exc:
        print(f"  [bertscore_err] {exc}")
        return {k: float("nan") for k in _default}

def _semantic_sim(reference, candidate):
    if not str(reference).strip() or not str(candidate).strip():
        return float("nan")
    v = _embed_batch_sc([str(reference), str(candidate)])
    v = v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-9)
    return float(round(v[0] @ v[1], 4))

pos["paperqa_clean"] = pos["paperqa_answer"].map(_strip_citations)
res = [({"bertscore_p": float("nan"), "bertscore_r": float("nan"), "bertscore_f": float("nan")}
        if _is_invalid_answer_sc(r["paperqa_clean"])
        else _bertscore_sc(r["ground_truth"], r["paperqa_clean"]))
       for _, r in pos.iterrows()]
for k in ("bertscore_p", "bertscore_r", "bertscore_f"):
    pos[f"paperqa_{k}"] = [x[k] for x in res]
pos["paperqa_semantic_sim"] = [(float("nan") if _is_invalid_answer_sc(r["paperqa_clean"])
                                else _semantic_sim(r["ground_truth"], r["paperqa_clean"]))
                               for _, r in pos.iterrows()]
_n_bad = int(pos["paperqa_clean"].map(_is_invalid_answer_sc).sum())
if _n_bad:
    print(f"[EXCLUDED] paperqa: {_n_bad} empty/API-error answer(s) -> NaN (dropped from means)")
print("scored", len(pos), "questions")


scored 30 questions


## Cell 5b: size-matched scoring

PaperQA2, like MedRAG, is not length-constrained, while the T21 summary is capped at 150
words and the expert references are shorter still. Length alone moves these metrics: a
longer answer tends to raise recall and depress precision independently of retrieval
quality. This repeats the scoring on answers trimmed to the reference word count, keeping
each answer's most relevant sentences, so no arm gains from verbosity. Report both.


In [6]:
# ---- Cell 5b: size-matched ----
def _size_match(reference, candidate):
    ref, cand = str(reference or "").strip(), str(candidate or "").strip()
    if not ref or not cand:
        return cand
    target = len(ref.split())
    sents = _split_sents_sc(cand)
    if len(sents) <= 1 or len(cand.split()) <= target:
        return cand
    vecs = _embed_batch_sc([ref] + sents)
    vecs = vecs / np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-9)
    sims = vecs[1:] @ vecs[0]
    keep, used = set(), 0
    for idx in np.argsort(-sims):
        w = len(sents[idx].split())
        if used and used + w > target:
            continue
        keep.add(int(idx)); used += w
        if used >= target:
            break
    if not keep:
        keep = {int(np.argmax(sims))}
    return " ".join(sents[i] for i in sorted(keep))

pos["paperqa_answer_sm"] = [_size_match(r["ground_truth"], r["paperqa_clean"])
                            for _, r in pos.iterrows()]
res_sm = [_bertscore_sc(r["ground_truth"], r["paperqa_answer_sm"]) for _, r in pos.iterrows()]
for k in ("bertscore_p", "bertscore_r", "bertscore_f"):
    pos[f"paperqa_sm_{k}"] = [x[k] for x in res_sm]

print(f"full length   BERTScore F {pos['paperqa_bertscore_f'].mean():.4f} "
      f"({pos['paperqa_clean'].str.split().str.len().mean():.0f} words)")
print(f"size-matched  BERTScore F {pos['paperqa_sm_bertscore_f'].mean():.4f} "
      f"({pos['paperqa_answer_sm'].str.split().str.len().mean():.0f} words)")


full length   BERTScore F 0.5324 (133 words)
size-matched  BERTScore F 0.5744 (61 words)


## Cell 6: results, and comparison with MedRAG

If notebook 17 has been run, its per-question scores are loaded so the two baselines can
be compared directly on the same questions with a paired test. Paired is correct here
because both arms answered the identical 30 items.

A null result at n=30 means no difference was detected at this sample size, not that the
systems are equivalent. That distinction is exactly what reviewer 1 raised in major
comment 4, so state the effect size alongside any p value.


In [7]:
# ---- Cell 6: results ----
metrics = ["bertscore_p", "bertscore_r", "bertscore_f", "semantic_sim"]
rows = {"paperqa": [pos[f"paperqa_{m}"].mean() for m in metrics]}
rows["paperqa (size-matched F)"] = [np.nan, np.nan, pos["paperqa_sm_bertscore_f"].mean(), np.nan]

MEDRAG_CSV = ANALYSIS / "output" / "medrag_benchmark" / "medrag_only_results.csv"
med = None
if MEDRAG_CSV.exists():
    med = pd.read_csv(MEDRAG_CSV)[["SLNo"] + [f"medrag_{m}" for m in metrics]]
    pos_m = pos.merge(med, on="SLNo", how="left")
    rows["medrag"] = [pos_m[f"medrag_{m}"].mean() for m in metrics]
else:
    pos_m = pos
    print("notebook 17 results not found; skipping the MedRAG comparison")

summary = pd.DataFrame(rows, index=metrics).round(4)
print("mean scores over 30 positive-control questions")
print(summary.to_string())

if med is not None:
    from scipy.stats import wilcoxon
    a = pos_m["paperqa_bertscore_f"].astype(float)
    b = pos_m["medrag_bertscore_f"].astype(float)
    ok = a.notna() & b.notna()
    stat, p = wilcoxon(a[ok], b[ok])
    print(f"\npaired Wilcoxon, PaperQA2 vs MedRAG, BERTScore F (n={int(ok.sum())})")
    print(f"  mean difference (PaperQA2 minus MedRAG): {float((a[ok]-b[ok]).mean()):+.4f}")
    print(f"  statistic={stat:.1f}  p={p:.4f}")
    print("  a null result at n=30 means not detected, not equivalent")

pos.to_csv(OUT_DIR / "paperqa_benchmark_results.csv", index=False)
summary.to_csv(OUT_DIR / "paperqa_benchmark_summary.csv")
print(f"\nsaved to {OUT_DIR}")


notebook 17 results not found; skipping the MedRAG comparison
mean scores over 30 positive-control questions
              paperqa  paperqa (size-matched F)
bertscore_p    0.5089                       NaN
bertscore_r    0.5728                       NaN
bertscore_f    0.5324                    0.5744
semantic_sim   0.6594                       NaN

saved to /path/to/project/T21_research_assistant_v7/analysis/output/paperqa_benchmark


## Cell 7: per-question detail and grounding audit

The failure pattern is the useful part for the response to reviewers. An aggregate mean
says a difference exists; the per-question view says where it comes from. On the MedRAG
arm the weakest items were a negative-polarity question (which cancers are people with
DS LESS prone to) and specific mechanistic queries, both cases where the answer sits in
Results or Methods rather than background prose. Check whether the same items are weak
here, and whether the T21 arm handles them better.


In [8]:
# ---- Cell 7: per-question ----
cols = ["SLNo", "question", "paperqa_bertscore_f"]
if med is not None:
    cols.append("medrag_bertscore_f")
detail = pos_m[cols].copy()
if med is not None:
    detail["delta"] = (detail["paperqa_bertscore_f"] - detail["medrag_bertscore_f"]).round(4)
    detail = detail.sort_values("delta")
print(detail.to_string(index=False, max_colwidth=56))

print("\nweakest 5 questions for PaperQA2")
for _, r in pos.nsmallest(5, "paperqa_bertscore_f").iterrows():
    print(f"  {r['paperqa_bertscore_f']:.3f}  {str(r['question'])[:76]}")

k0 = str(pos.iloc[0]["SLNo"])
rec = answers.get(k0, {})
print(f"\nGROUNDING AUDIT for SLNo {k0}")
print("  contexts used :", rec.get("n_contexts", "-"))
print("  papers        :", ", ".join(rec.get("papers", [])[:8]))
print("  answer        :", rec.get("answer", "")[:400])


SLNo                                                 question  paperqa_bertscore_f
  Q1 Which clinical conditions are people with Down syndro...               0.4806
  Q2 Which clinical conditions are people with Down syndro...               0.4873
  Q3 Why do people with Down syndrome tend to have hyperac...               0.5079
  Q4 Why are people with Down syndrome at high risk of dev...               0.5423
  Q5 Do individuals with Down syndrome have overactive thy...               0.7192
  Q6 Which interferon receptors are triplicated in Down sy...               0.6493
  Q7        How is Stat1 signaling affected in Down syndrome?               0.5271
  Q8 Which circulating immune cell is consistently deplete...               0.4831
  Q9 What are some advantages and caveats of the Ts65Dn an...               0.5915
 Q10 Are individuals with Down syndrome more prone to high...               0.7161
 Q11 What neurodegeneration is associated with Down syndrome?               0.5629
 Q12

---
## Notes for the manuscript

Describe this arm as "PaperQA2, run over the same corpus and generator, using its
non-agentic Docs pipeline". Three deviations from the shipped defaults must be stated,
all documented in analysis/benchmarks/paperqa_adapter.py:

  1. All three LLM roles were repointed from GPT-4o to nemotron-3-super, so the arm
     tests architecture rather than model quality. With EMBED_PROVIDER="t21" the
     embedder is also T21's, verified by HTTP capture: 16 calls to
     ollama.com/v1/chat/completions (model=nemotron-3-super) and 3 to
     api.deepinfra.com/v1/openai/embeddings
     (model=nvidia/llama-nemotron-embed-vl-1b-v2), with no OpenAI traffic.
     If you instead run EMBED_PROVIDER="openai", say so explicitly, and note the
     asymmetry with the MedRAG arm, which is run as published.
  2. The agentic entry point was not used, because paper-qa 2026.8.12 is incompatible
     with fhlmi 1.0.7 (get_router was removed upstream). The deterministic Docs pipeline
     was used instead, which also removes agent-induced run-to-run variance.
  3. Document metadata lookups (use_doc_details) were disabled; the citation header is
     already embedded at the top of each text file, and the lookups are slow and
     rate-limited.

Corpus note: PaperQA2 received the same papers as plain text rather than the downloaded
PDFs. Only about 80 percent of the corpus has a fetchable PDF, so a PDF-based run would
have silently dropped a fifth of the papers, and PDF text extraction would have
introduced a second confound. Text files are also deterministic where PDF parsers are
not.

Measured API usage, verified by instrumenting the HTTP calls: one question over three
papers produced 16 calls to ollama.com/v1/chat/completions and 4 to
api.openai.com/v1/embeddings, with no Crossref or Semantic Scholar traffic. The
per-chunk evidence summarisation is what makes PaperQA2 markedly more LLM-intensive
than MedRAG, which used a single call per question.
